# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OsaidKamran/FLYRANK.AI-SUMMER-INTERNSHIP-MACHINE-LEARNING_UPDATED/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Health Score Feature Importance (p.27). The paper's Random Forest model predicts health_score and finds that Average Position (43%), Impressions (32%), Scroll Depth (15%), and CTR (8%) dominate feature importance, while every other feature — content age, word count, days visible, search volume, CPC, competition — scores approximately 0%. The paper itself flags this as a limitation, since health_score is explicitly defined as a weighted sum of impressions, position, CTR, and scroll depth (Methodology, p.5). The methodology question worth asking is not whether this circularity exists — the paper already discloses it — but how large the gap is: what does the model's predictive power look like with only the four independent, non-zero-adjacent features included, and the four label-constituent features removed entirely? That number, rather than the current 43/32/15/8% breakdown, is the one that would tell a reader whether health score carries any signal beyond its own arithmetic definition.

Finding 2 — Growth Prediction Holdout Accuracy (p.28). The paper reports 71% holdout accuracy for a logistic regression predicting growth versus decline, using an 80/20 split, across a dataset spanning 57 brands. The label itself is defined from a 30-day-versus-previous-30-day impression trend threshold. The Methodology section names content age as a confounding variable but does not state whether the 80/20 split was grouped by brand or drawn at random across all 341,701 rows. Because pages within the same brand likely share formatting, publishing cadence, and template choices, a random split risks letting the model partially learn brand identity rather than generalizable growth signal — the same risk this notebook's Section 2 tests directly on client identity in my own capstone. The constructive follow-up question: would the 71% figure hold if evaluated on brands entirely held out from training, the same way a client-grouped split would be evaluated?

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-running the ML-08 pipeline (same 120,475-row feature frame, same 14 features, same PCA/Autoencoder architecture, same k=8) under a plain random 80/20 split instead of the client-grouped split reveals a measurable honesty gap. Test-set silhouette rose from 0.2669 to 0.3493 for the PCA baseline (+0.0824) and from 0.2413 to 0.3498 for the Autoencoder (+0.1086) under the random split — both models appear meaningfully stronger once the split stops respecting client boundaries.

The more informative evidence is the train-to-test gap itself, not just the absolute numbers. Under the grouped split, silhouette drops by roughly 0.10 (PCA) and 0.03 (Autoencoder) from train to test — a real generalization gap consistent with evaluating on genuinely unseen clients. Under the random split, that gap disappears almost entirely (train and test silhouette differ by 0.003 and 0.002 respectively). The client-overlap check explains why: the grouped split guarantees zero shared clients between train and test, while the random split leaves 100% of test-set clients also present in the training set (40 of 40). The apparent "improvement" under random split is best read as the model partially re-recognizing client-level structure it already saw during training, rather than genuinely stronger archetype separation.

One result did not follow the expected direction and is reported rather than omitted: the Autoencoder's test reconstruction MSE was slightly higher under the random split (0.0148) than the grouped split (0.0131), the opposite of what the silhouette pattern would predict. This gap is small relative to the model's overall MSE and is more plausibly attributable to a different train/test row composition than to a systematic effect, but it is noted here as a genuinely mixed result rather than smoothed into the main narrative.

These results are decision-support for validation methodology, not a claim about which silhouette number is "true." The client-grouped split used throughout the ML-08 capstone pipeline is the split that measures whether discovered archetypes generalize to clients the model has never seen — which is the actual question the capstone needs answered, and the one the random-split comparison shows a plain 80/20 split cannot honestly answer for this dataset.

In [1]:
# ===== PART 1: SETUP AND DATA LOAD =====
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

import numpy as np
import pandas as pd
from pathlib import Path
import duckdb
import joblib
import warnings

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, silhouette_score

warnings.filterwarnings('ignore', category=FutureWarning)

# Exact values extracted from W05 Model Notebook
RANDOM_STATE = 42

print("--- REUSED PARAMETERS FROM ML-08 (W05) ---")
print(f"random_state: {RANDOM_STATE}")
print("StandardScaler: Default configuration")
print("PCA n_components: 8")
print("KMeans k: 8 (Fixed for this task)")
print("MLPRegressor architecture: hidden_layer_sizes=(16, 8, 16), activation='relu', early_stopping=True")
print("Bottleneck Extraction: Manual NumPy forward pass (ReLU) through coefs_[0..1] and intercepts_[0..1]")
print(f"Sampled Silhouette: sample_size=10000, random_state={RANDOM_STATE}")
print("------------------------------------------\n")

# Safely resolve repo root
current_dir = Path.cwd().resolve()
repo_root = current_dir
while not (repo_root / 'work').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

cache_path = repo_root / "work" / "outputs" / "w05_features_2026_03_half_floor10.parquet"
if not cache_path.exists():
    raise FileNotFoundError(f"Expected to find feature frame at: {cache_path}")

# Mandatory DuckDB file read (no pandas read_parquet)
con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{cache_path}')").df()

feature_cols = [
    'impressions_first_half', 'clicks_first_half', 'ctr', 'avg_position', 
    'engagement_rate', 'has_ga4_coverage', 'search_volume', 'competition', 
    'backlinks', 'word_count', 'char_count', 'has_backlink_data', 
    'has_word_count_data', 'content_age_days'
]

# Cast to float to prevent mixed-type comparison issues in pandas 3.0.5+
df[feature_cols] = df[feature_cols].astype(float)


# ===== PART 2: THE TWO SPLIT CONDITIONS =====

# Condition A: GROUPED Split (Client-aware, reproducing W05)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx_g, test_idx_g = next(gss.split(df, groups=df['client_hash_id']))
train_df_g = df.iloc[train_idx_g].copy().reset_index(drop=True)
test_df_g = df.iloc[test_idx_g].copy().reset_index(drop=True)

# Condition B: RANDOM Split (Row-level, ignoring groups)
train_df_r, test_df_r = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=None)
train_df_r = train_df_r.copy().reset_index(drop=True)
test_df_r = test_df_r.copy().reset_index(drop=True)


# ===== PART 3: MODEL PIPELINE EXECUTION =====

def run_pipeline(train_df, test_df, split_name):
    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    
    # 1. Standardize
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 2. PCA Baseline
    pca = PCA(n_components=8, random_state=RANDOM_STATE)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    with joblib.parallel_backend('threading'):
        km_pca = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init='auto').fit(X_train_pca)
        
    pca_train_recon = pca.inverse_transform(X_train_pca)
    pca_test_recon = pca.inverse_transform(X_test_pca)
    pca_train_mse = mean_squared_error(X_train_scaled, pca_train_recon)
    pca_test_mse = mean_squared_error(X_test_scaled, pca_test_recon)
    
    # Using the exact sampled-silhouette method isolated in w05's scan phase
    pca_train_sil = silhouette_score(X_train_pca, km_pca.labels_, sample_size=10000, random_state=RANDOM_STATE)
    pca_test_sil = silhouette_score(X_test_pca, km_pca.predict(X_test_pca), sample_size=10000, random_state=RANDOM_STATE)
    
    # 3. Autoencoder (MLPRegressor)
    mlp = MLPRegressor(
        hidden_layer_sizes=(16, 8, 16),
        activation='relu',
        random_state=RANDOM_STATE,
        early_stopping=True
    )
    mlp.fit(X_train_scaled, X_train_scaled)
    
    ae_train_recon = mlp.predict(X_train_scaled)
    ae_test_recon = mlp.predict(X_test_scaled)
    ae_train_mse = mean_squared_error(X_train_scaled, ae_train_recon)
    ae_test_mse = mean_squared_error(X_test_scaled, ae_test_recon)
    
    # Extract bottleneck simulating ReLU logic from w05
    def extract_bottleneck(X_in, mlp_model):
        h1 = np.maximum(0, np.dot(X_in, mlp_model.coefs_[0]) + mlp_model.intercepts_[0])
        bottleneck = np.maximum(0, np.dot(h1, mlp_model.coefs_[1]) + mlp_model.intercepts_[1])
        return bottleneck
        
    X_train_ae = extract_bottleneck(X_train_scaled, mlp)
    X_test_ae = extract_bottleneck(X_test_scaled, mlp)
    
    with joblib.parallel_backend('threading'):
        km_ae = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init='auto').fit(X_train_ae)
        
    ae_train_sil = silhouette_score(X_train_ae, km_ae.labels_, sample_size=10000, random_state=RANDOM_STATE)
    ae_test_sil = silhouette_score(X_test_ae, km_ae.predict(X_test_ae), sample_size=10000, random_state=RANDOM_STATE)
    
    return [
        {'split_type': split_name, 'model': 'PCA', 'train_recon_mse': pca_train_mse, 'test_recon_mse': pca_test_mse, 'train_silhouette': pca_train_sil, 'test_silhouette': pca_test_sil},
        {'split_type': split_name, 'model': 'Autoencoder', 'train_recon_mse': ae_train_mse, 'test_recon_mse': ae_test_mse, 'train_silhouette': ae_train_sil, 'test_silhouette': ae_test_sil}
    ]

print("Executing pipeline on GROUPED split condition...")
results_g = run_pipeline(train_df_g, test_df_g, 'GROUPED')

print("Executing pipeline on RANDOM split condition...")
results_r = run_pipeline(train_df_r, test_df_r, 'RANDOM')


# ===== PART 4: OUTPUT AND MECHANISM CHECK =====

results_df = pd.DataFrame(results_g + results_r)

print("\n--- MODEL EVALUATION COMPARISON ---")
print(results_df.to_string(index=False))

# Client Overlap Check
print("\n--- CLIENT SPLIT OVERLAP CHECK ---")

train_clients_g = set(train_df_g['client_hash_id'])
test_clients_g = set(test_df_g['client_hash_id'])
n_test_clients_g = len(test_clients_g)
n_overlap_g = len(train_clients_g.intersection(test_clients_g))
frac_g = n_overlap_g / n_test_clients_g if n_test_clients_g > 0 else 0

print("[GROUPED SPLIT]")
print(f"Train Set: {len(train_df_g)} rows, {len(train_clients_g):>2} unique clients")
print(f"Test Set:  {len(test_df_g)} rows, {n_test_clients_g:>2} unique clients")
print(f"Overlap:   {n_overlap_g:>2} clients in both ({frac_g*100:.1f}%)")

train_clients_r = set(train_df_r['client_hash_id'])
test_clients_r = set(test_df_r['client_hash_id'])
n_test_clients_r = len(test_clients_r)
n_overlap_r = len(train_clients_r.intersection(test_clients_r))
frac_r = n_overlap_r / n_test_clients_r if n_test_clients_r > 0 else 0

print("\n[RANDOM SPLIT]")
print(f"Train Set: {len(train_df_r)} rows, {len(train_clients_r):>2} unique clients")
print(f"Test Set:  {len(test_df_r)} rows, {n_test_clients_r:>2} unique clients")
print(f"Overlap:   {n_overlap_r:>2} clients in both ({frac_r*100:.1f}%)")

# Gap Analysis
print("\n--- TEST SILHOUETTE GAP SUMMARY (RANDOM minus GROUPED) ---")
pca_gap = results_df.loc[(results_df['split_type']=='RANDOM') & (results_df['model']=='PCA'), 'test_silhouette'].values[0] - \
          results_df.loc[(results_df['split_type']=='GROUPED') & (results_df['model']=='PCA'), 'test_silhouette'].values[0]
          
ae_gap = results_df.loc[(results_df['split_type']=='RANDOM') & (results_df['model']=='Autoencoder'), 'test_silhouette'].values[0] - \
         results_df.loc[(results_df['split_type']=='GROUPED') & (results_df['model']=='Autoencoder'), 'test_silhouette'].values[0]
         
print(f"PCA Silhouette Gap:         {pca_gap:+.4f}")
print(f"Autoencoder Silhouette Gap: {ae_gap:+.4f}")
print("A positive gap measures how much higher cluster separation appears when client identity leaks across the split, supporting the choice of a grouped split for directional evaluation.")

--- REUSED PARAMETERS FROM ML-08 (W05) ---
random_state: 42
StandardScaler: Default configuration
PCA n_components: 8
KMeans k: 8 (Fixed for this task)
MLPRegressor architecture: hidden_layer_sizes=(16, 8, 16), activation='relu', early_stopping=True
Bottleneck Extraction: Manual NumPy forward pass (ReLU) through coefs_[0..1] and intercepts_[0..1]
Sampled Silhouette: sample_size=10000, random_state=42
------------------------------------------

Executing pipeline on GROUPED split condition...
Executing pipeline on RANDOM split condition...

--- MODEL EVALUATION COMPARISON ---
split_type       model  train_recon_mse  test_recon_mse  train_silhouette  test_silhouette
   GROUPED         PCA         0.155174        0.263669          0.369493         0.266921
   GROUPED Autoencoder         0.004782        0.013076          0.273094         0.241270
    RANDOM         PCA         0.160997        0.146450          0.346281         0.349306
    RANDOM Autoencoder         0.015542        0.01484

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [2]:
# ===== STEP 0: READ AND CONFIRM CONTEXT =====
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['LOKY_MAX_CPU_COUNT'] = '1'

import numpy as np
import pandas as pd
from pathlib import Path
import duckdb
import joblib
import warnings
import json
import re
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore', category=FutureWarning)

RANDOM_STATE = 42

print("--- STEP 0: CONFIRMED ML-08 (W05) PARAMETERS & SQL CHECKS ---")
print(f"random_state: {RANDOM_STATE}")
print("StandardScaler: Default configuration")
print("PCA n_components: 8")
print("KMeans k: 8 (Fixed for this task)")
print("MLPRegressor architecture: hidden_layer_sizes=(16, 8, 16), activation='relu', early_stopping=True")
print("Bottleneck Extraction: Manual NumPy forward pass (ReLU) through coefs_[0..1] and intercepts_[0..1]")
print(f"Sampled Silhouette: sample_size=10000, random_state={RANDOM_STATE}")

# Safely resolve repo root
current_dir = Path.cwd().resolve()
repo_root = current_dir
while not (repo_root / 'work').exists() and repo_root.parent != repo_root:
    repo_root = repo_root.parent

print("\n--- ACTUAL SQL W05 TEXT INSPECTION ---")
w05_path = repo_root / "work" / "notebooks" / "w05_model.ipynb"
if not w05_path.exists():
    raise FileNotFoundError(f"Missing W05 notebook at: {w05_path}")

# Parse JSON and extract the source text from all code cells
with open(w05_path, 'r', encoding='utf-8') as f:
    w05_data = json.load(f)

w05_code_text = ""
for cell in w05_data.get('cells', []):
    if cell.get('cell_type') == 'code':
        source = cell.get('source', [])
        if isinstance(source, list):
            w05_code_text += "".join(source) + "\n"
        else:
            w05_code_text += source + "\n"

# 1. Anchor Date Check
fixed_anchor_match = re.search(r"DATE\s+'2026-03-15'", w05_code_text, re.IGNORECASE)
moving_ref_match = re.search(r"MAX\(\s*[^)]*report_date[^)]*\)", w05_code_text, re.IGNORECASE)

print("Anchor Date Analysis:")
if fixed_anchor_match:
    print(f"  Fixed anchor matched text: {fixed_anchor_match.group(0)}")
else:
    print("  Fixed anchor matched text: NOT FOUND")

if moving_ref_match:
    print(f"  Moving reference matched text: {moving_ref_match.group(0)}")
else:
    print("  Moving reference matched text: NOT FOUND")

if fixed_anchor_match and not moving_ref_match:
    print("  Anchor Date Check: PASS")
else:
    print("  WARNING: Anchor date is not a fixed literal, or a moving reference like MAX(report_date) was found!")
    print("  Anchor Date Check: FAIL")

# 2. Window Boundary Check
valid_window_match = re.search(r"report_date\s*<=\s*DATE\s*'2026-03-15'", w05_code_text, re.IGNORECASE)
leak_window_match = re.search(r"report_date\s*>\s*DATE\s*'2026-03-15'|EXTRACT\(\s*DAY\s+FROM[^)]+\)\s*>\s*15", w05_code_text, re.IGNORECASE)

print("\nWindow Boundary Analysis:")
if valid_window_match:
    print(f"  Valid window boundary matched text: {valid_window_match.group(0)}")
else:
    print("  Valid window boundary matched text: NOT FOUND")

if leak_window_match:
    print(f"  Leaky window boundary matched text: {leak_window_match.group(0)}")
else:
    print("  Leaky window boundary matched text: NOT FOUND")

if valid_window_match and not leak_window_match:
    print("  Window Boundary Check: PASS")
else:
    print("  WARNING: Window boundary is not securely bounded to days 1-15, or future window data (days 16-31) is referenced!")
    print("  Window Boundary Check: FAIL")

print("-------------------------------------------------------------\n")

cache_path = repo_root / "work" / "outputs" / "w05_features_2026_03_half_floor10.parquet"
if not cache_path.exists():
    raise FileNotFoundError(f"Missing cache at: {cache_path}")

con = duckdb.connect()
df = con.execute(f"SELECT * FROM read_parquet('{cache_path}')").df()

feature_cols = [
    'impressions_first_half', 'clicks_first_half', 'ctr', 'avg_position', 
    'engagement_rate', 'has_ga4_coverage', 'search_volume', 'competition', 
    'backlinks', 'word_count', 'char_count', 'has_backlink_data', 
    'has_word_count_data', 'content_age_days'
]

# Ensure consistent typing
df[feature_cols] = df[feature_cols].astype(float)


# ===== PART A: CHECKLIST VERIFICATION =====
print("--- PART A: LEAKAGE CHECKLIST VERIFICATION ---")

# 1. Assert context IDs are not features
check_1_pass = ('client_hash_id' not in feature_cols) and ('content_hash_id' not in feature_cols)
print(f"1. Context IDs safely excluded from the model feature set: {'PASS' if check_1_pass else 'FAIL'}")

# 2. Hardcoded schemas check for product flags
dim_content_cols = [
    'client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 
    'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 
    'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 
    'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 
    'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 
    'is_published', 'is_deleted'
]
daily_cols = [
    'report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 
    'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 
    'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 
    'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 
    'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 
    'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month'
]

all_cols = dim_content_cols + daily_cols
forbidden_strings = ["health_score", "flag", "quick_win", "needs_"]
check_2_pass = not any(any(forb in col for col in all_cols) for forb in forbidden_strings)
print(f"2. No product-decision flags (health_score, flag, quick_win, needs_) exist in warehouse schemas: {'PASS' if check_2_pass else 'FAIL'}")

# 3. Fixed anchor and window pass
anchor_check_pass = bool(fixed_anchor_match) and not bool(moving_ref_match)
window_check_pass = bool(valid_window_match) and not bool(leak_window_match)
check_3_pass = anchor_check_pass and window_check_pass
print(f"3. Fixed-date anchor and first-half-only window confirmed via actual SQL inspection: {'PASS' if check_3_pass else 'FAIL'}")


# ===== PART B: POSITIVE CONTROL (CLIENT ID LEAK INJECTION) =====
print("\n--- PART B: POSITIVE CONTROL (CLIENT ID LEAK INJECTION) ---")

# 1. Grouped Split (W05 identical replication)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df['client_hash_id']))
train_df = df.iloc[train_idx].copy().reset_index(drop=True)
test_df = df.iloc[test_idx].copy().reset_index(drop=True)

# 2. Build the CONTAMINATED feature set
# One-hot encode client identity on train, then reindex on test to keep feature columns perfectly aligned
train_dummies = pd.get_dummies(train_df['client_hash_id'], prefix='client')
test_dummies = pd.get_dummies(test_df['client_hash_id'], prefix='client')
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train_contam = pd.concat([train_df[feature_cols], train_dummies], axis=1).astype(float)
test_contam = pd.concat([test_df[feature_cols], test_dummies], axis=1).astype(float)

num_unique_train_clients = train_df['client_hash_id'].nunique()
expected_features = 14 + num_unique_train_clients
actual_features = train_contam.shape[1]

print(f"Contaminated feature count: {actual_features}")
print(f"Matches 14 legitimate features + {num_unique_train_clients} unique train clients: {'Yes' if actual_features == expected_features else 'No'}")

# 3. Run IDENTICAL Pipeline on Contaminated Data
scaler_c = StandardScaler()
X_train_c = scaler_c.fit_transform(train_contam.values)
X_test_c = scaler_c.transform(test_contam.values)

# PCA Branch
pca_c = PCA(n_components=8, random_state=RANDOM_STATE)
X_train_pca_c = pca_c.fit_transform(X_train_c)
X_test_pca_c = pca_c.transform(X_test_c)

with joblib.parallel_backend('threading'):
    km_pca_c = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init='auto').fit(X_train_pca_c)

pca_test_sil_c = silhouette_score(X_test_pca_c, km_pca_c.predict(X_test_pca_c), sample_size=10000, random_state=RANDOM_STATE)

# Autoencoder Branch
mlp_c = MLPRegressor(hidden_layer_sizes=(16, 8, 16), activation='relu', random_state=RANDOM_STATE, early_stopping=True)
mlp_c.fit(X_train_c, X_train_c)

def extract_bottleneck(X_in, mlp_model):
    h1 = np.maximum(0, np.dot(X_in, mlp_model.coefs_[0]) + mlp_model.intercepts_[0])
    bottleneck = np.maximum(0, np.dot(h1, mlp_model.coefs_[1]) + mlp_model.intercepts_[1])
    return bottleneck

X_train_ae_c = extract_bottleneck(X_train_c, mlp_c)
X_test_ae_c = extract_bottleneck(X_test_c, mlp_c)

with joblib.parallel_backend('threading'):
    km_ae_c = KMeans(n_clusters=8, random_state=RANDOM_STATE, n_init='auto').fit(X_train_ae_c)

ae_test_sil_c = silhouette_score(X_test_ae_c, km_ae_c.predict(X_test_ae_c), sample_size=10000, random_state=RANDOM_STATE)

# 4. Comparison Table (Hardcoded CLEAN rows from Section 2)
clean_pca_sil = 0.2669
clean_ae_sil = 0.2413

compare_data = [
    {'condition': 'CLEAN', 'model': 'PCA', 'test_silhouette': clean_pca_sil},
    {'condition': 'CLEAN', 'model': 'Autoencoder', 'test_silhouette': clean_ae_sil},
    {'condition': 'CONTAMINATED', 'model': 'PCA', 'test_silhouette': pca_test_sil_c},
    {'condition': 'CONTAMINATED', 'model': 'Autoencoder', 'test_silhouette': ae_test_sil_c}
]
compare_df = pd.DataFrame(compare_data)

print("\n--- LEAKAGE INJECTION RESULTS ---")
print(compare_df.to_string(index=False))

# 5. Delta computations
pca_delta = pca_test_sil_c - clean_pca_sil
ae_delta = ae_test_sil_c - clean_ae_sil
print("\n--- DELTA (CONTAMINATED minus CLEAN) ---")
print(f"PCA Silhouette Delta:         {pca_delta:+.4f}")
print(f"Autoencoder Silhouette Delta: {ae_delta:+.4f}")
print("Observation: Injecting client identity mathematically inflates the silhouette score. This is consistent with models isolating client-specific groupings rather than generalizing pure content archetypes.")

# 6. Final Fresh-Load Integrity Check
df_fresh = con.execute(f"SELECT * FROM read_parquet('{cache_path}')").df()
raw_cols = set(df_fresh.columns)
expected_raw_cols = set(feature_cols + ['content_hash_id', 'client_hash_id'])

is_exact_match = (raw_cols == expected_raw_cols)
print(f"\nFinal Check: Freshly loaded production frame has exactly the {len(expected_raw_cols)} expected columns and zero client-derived dummy features leaked into storage: {'PASS' if is_exact_match else 'FAIL'}")

--- STEP 0: CONFIRMED ML-08 (W05) PARAMETERS & SQL CHECKS ---
random_state: 42
StandardScaler: Default configuration
PCA n_components: 8
KMeans k: 8 (Fixed for this task)
MLPRegressor architecture: hidden_layer_sizes=(16, 8, 16), activation='relu', early_stopping=True
Bottleneck Extraction: Manual NumPy forward pass (ReLU) through coefs_[0..1] and intercepts_[0..1]
Sampled Silhouette: sample_size=10000, random_state=42

--- ACTUAL SQL W05 TEXT INSPECTION ---
Anchor Date Analysis:
  Fixed anchor matched text: DATE '2026-03-15'
  Moving reference matched text: NOT FOUND
  Anchor Date Check: PASS

Window Boundary Analysis:
  Valid window boundary matched text: report_date <= DATE '2026-03-15'
  Leaky window boundary matched text: NOT FOUND
  Window Boundary Check: PASS
-------------------------------------------------------------

--- PART A: LEAKAGE CHECKLIST VERIFICATION ---
1. Context IDs safely excluded from the model feature set: PASS
2. No product-decision flags (health_score, flag,

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.